In [54]:
from pathlib import Path

COLAB_CWD = Path("/content")
CWD = Path.cwd()

def is_colab():
    return CWD == COLAB_CWD

IN_COLAB = False

if is_colab():
    IN_COLAB = True
    !git clone "https://github.com/fbocchi/show-and-tell.git"
    %cd "show-and-tell"

In [55]:
from models.loss import loss_that_ignores_padding

loss_that_ignores_padding

<function models.loss.loss_that_ignores_padding(y_true, y_pred)>

In [56]:
if IN_COLAB:
    %cd "notebooks"
    CWD = Path.cwd()

print(CWD)

C:\Users\franc\PycharmProjects\PythonProject\notebooks


In [57]:
import os
import pandas as pd

train_df = pd.read_csv(f"{CWD}/../flickr8k/captions_train.csv")

image_paths = train_df["image"].apply(lambda x: os.path.join(f"{CWD}/../flickr8k/Images", x)).to_numpy()
captions = train_df["caption"].apply(lambda x: f"[START] {x} [END]").to_numpy()

print(captions)

['[START] a child in a pink dress is climbing up a set of stairs in an entry way [END]'
 '[START] a girl going into a wooden building [END]'
 '[START] a little girl climbing into a wooden playhouse [END]' ...
 '[START] a person in a red shirt climbing up a rock face covered in assist handles [END]'
 '[START] a rock climber in a red shirt [END]'
 '[START] a rock climber practices on a rock climbing wall [END]']


In [58]:
import json

import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

with open(f"{CWD}/../config/config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

vectorizer = TextVectorization(
    max_tokens=None,
    standardize=None,
    split="whitespace",
    output_mode="int",
    output_sequence_length=config["max_len"] + 2,
    pad_to_max_tokens=False,
    vocabulary=config["vocab"],
)

vocab = vectorizer.get_vocabulary()
print(len(vocab))
print(vocab)

def create_sample(image_path, caption):
    image = load_image(image_path)
    in_caption, target_caption = split_caption_for_teacher_forcing(caption)
    in_caption = vectorizer(in_caption)
    target_caption = vectorizer(target_caption)
    return (image, in_caption), target_caption

def load_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, 3)
    img = tf.image.resize(img, (299, 299))
    return tf.keras.applications.inception_v3.preprocess_input(img)

def split_caption_for_teacher_forcing(caption):
    tokens = tf.strings.split(caption, sep=" ")

    tok_in_caption = tokens[:-1]
    tok_target_caption = tokens[1:]

    in_caption = tf.strings.reduce_join(
        tok_in_caption,
        separator=" "
    )

    target_caption = tf.strings.reduce_join(
        tok_target_caption,
        separator=" "
    )

    return in_caption, target_caption

2905
['', '[UNK]', '[START]', '[END]', 'a', 'in', 'the', 'on', 'is', 'and', 'dog', 'with', 'man', 'of', 'two', 'white', 'black', 'boy', 'are', 'woman', 'girl', 'to', 'wearing', 'at', 'people', 'water', 'red', 'young', 'brown', 'an', 'his', 'blue', 'dogs', 'running', 'through', 'playing', 'while', 'shirt', 'down', 'ball', 'little', 'standing', 'grass', 'person', 'child', 'snow', 'jumping', 'front', 'over', 'three', 'sitting', 'holding', 'field', 'small', 'group', 'green', 'up', 'large', 'by', 'one', 'yellow', 'her', 'children', 'walking', 'men', 'into', 'beach', 'air', 'near', 'mouth', 'jumps', 'for', 'another', 'street', 'runs', 'its', 'from', 'riding', 'stands', 'as', 'bike', 'girls', 'outside', 'other', 'rock', 'out', 'play', 'off', 'orange', 'pink', 'looking', 'next', 'player', 'pool', 'their', 'camera', 'boys', 'hat', 'jacket', 'women', 'around', 'behind', 'some', 'background', 'toy', 'dirt', 'soccer', 'dressed', 'sits', 'has', 'mountain', 'walks', 'wall', 'crowd', 'along', 'plays'

In [59]:
dataset = tf.data.Dataset.from_tensor_slices((image_paths, captions))
dataset = dataset.map(create_sample, num_parallel_calls=tf.data.AUTOTUNE)

dataset = dataset.shuffle(1000)
dataset = dataset.batch(64)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

In [60]:
import keras

model = keras.models.load_model(
    "../models/model.keras",
    custom_objects={
        "loss_that_ignores_padding": loss_that_ignores_padding
    }
)

model.fit(dataset, epochs=1)

model.save("../models/model_train.keras")


C:\Users\franc\PycharmProjects\PythonProject\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 26 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


601/601 ━━━━━━━━━━━━━━━━━━━━ 5027s 8s/step - loss: 4.8664
